# CS5491 — GNN+ normalization × depth sweep (Kaggle)

Runs the ZINC grid and rebuilds both PDF reports on **Kaggle Notebooks**
(free T4 GPU, ~30 GPU-hours/week, 12 h per session). Adapted from the Colab notebook.

## One-time setup
1. Verify your phone number in Kaggle account settings (required for GPU).
2. New Notebook → **Import Notebook** → upload this .ipynb file.
3. Settings → **Accelerator: GPU T4 x2**; Settings → **Internet: On**.

## Run order (do NOT change)
Cells 1–5: setup, install, GPU check, ZINC preflight, one-cell smoke test.
Cell 6: the full 20-cell grid. Cell 7: aggregate. Cells 8–9: rebuild PDFs and package.

Each cell prints a clear OK marker. **Stop and fix if a cell fails** — do not continue.

## About the epoch budget
EPOCHS = 30 in Cell 6 keeps the whole grid well within Kaggle's 12-hour session
(roughly 1.5–2 hours on T4). The report is generated with --epochs 30 so every
number and every sentence in the PDF matches what was actually run. Keep EPOCHS
identical in Cell 6 and Cell 8.

## If a session gets cut off (IMPORTANT)
When Kaggle reconnects a session, the working directory resets to
/kaggle/working and any in-progress cell is interrupted. **Every heavy cell
(6–9) now restores the correct directory itself**, so the recovery is simple:

1. Re-run **Cell 6 only** — `--skip-existing` instantly skips finished cells
   and resumes from the first unfinished one. No need to re-run Cells 1–5.
2. Each heavy cell re-clones the repo automatically if /kaggle/working was
   wiped. NOTE: a full wipe also erases results/ — you must re-run the grid.
   Click Save Version (Quick Save) often so this cannot happen.
3. Keep the same notebook. Files under /kaggle/working persist across
   reconnects for the **same notebook version**. If you import a fresh copy,
   download results/ from the Output panel first and re-upload it into
   /kaggle/working/repo/results/ before continuing.
4. File → **Save Version** before leaving, so results survive a runtime reset.


In [ ]:
# Cell 1 — clone / resume the repository, then force it to the latest commit.
import os, subprocess
from pathlib import Path

WORK = Path('/kaggle/working')
REPO = WORK / 'repo'
REPO_URL = 'https://github.com/1589893076/CS5491-GNNPlus-NormStudy.git'

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
# A saved /kaggle/working may hold an older clone. Hard-reset tracked files to
# origin/main so the 30-epoch notebook always runs the newest code. results/
# is gitignored and survives this reset, so partial progress is not lost.
subprocess.run(['git', 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=True)
head = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
print('working dir:', os.getcwd(), '| HEAD:', head)
assert (REPO / 'main.py').exists(), 'clone failed'


In [ ]:
# Cell 2 — install dependencies + compatibility patches
!pip -q install torch_geometric scikit-learn fsspec rdkit pytorch-lightning yacs torchmetrics networkx tensorboardX ogb wandb reportlab pypdf

import torch, os, shutil
from pathlib import Path

tv = torch.__version__.split('+')[0]
cuda_tag = 'cpu' if torch.version.cuda is None else 'cu' + torch.version.cuda.replace('.', '')
wheel_index = f'https://data.pyg.org/whl/torch-{tv}+{cuda_tag}.html'
print('torch', torch.__version__, '| wheel index', wheel_index)

!pip -q install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {wheel_index} || echo "compiled wheels unavailable -> fallback"

# Pure-PyTorch fallback for torch_scatter if no wheel matches.
try:
    import torch_scatter  # noqa
    print('torch_scatter: compiled extension available.')
except Exception as exc:
    root = Path(os.getcwd())
    shutil.copyfile(str(root / 'experiments' / 'torch_scatter_fallback.py'),
                    str(root / 'torch_scatter.py'))
    os.environ['PYTHONPATH'] = str(root) + os.pathsep + os.environ.get('PYTHONPATH', '')
    print('torch_scatter: pure-PyTorch fallback because:', exc)

# scikit-learn >= 1.6 removed ``squared`` from mean_squared_error (idempotent).
_logger = Path('GNNPlus/logger.py')
_src = _logger.read_text(encoding='utf-8')
_old = 'mean_squared_error(true, pred, squared=False)'
if _old in _src:
    _logger.write_text(_src.replace(_old, 'np.sqrt(mean_squared_error(true, pred))'),
                       encoding='utf-8')
    print('logger.py: patched sklearn RMSE call.')
else:
    print('logger.py: sklearn RMSE call already compatible.')


In [ ]:
# Cell 3 — GPU + real import chain check
import torch, os, subprocess, sys
assert torch.cuda.is_available(), 'No GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2.'
print('GPU:', torch.cuda.get_device_name(0))

_probe = subprocess.run(
    [sys.executable, '-c', 'import GNNPlus; print("GNNPlus import chain OK")'],
    cwd=os.getcwd(), capture_output=True, text=True)
print(_probe.stdout.strip())
if _probe.returncode:
    print(_probe.stderr[-2500:])
    raise RuntimeError('GNNPlus import chain failed. Fix before the sweep.')


In [ ]:
# Cell 4 — preflight: make sure ZINC can actually download
# Do NOT start the grid if this fails (all cells would fail at the loader).
!python -c "from torch_geometric.datasets import ZINC; d = ZINC(root='datasets/ZINC', subset=True, split='train'); print('ZINC train graphs:', len(d), '| sample:', d[0])" 


In [ ]:
# Cell 5 — one-cell smoke test (5 epochs, tagged so it never pollutes the table)
!python experiments/run_norm_study.py --norms batch --depths 4 --repeat 1 --epochs 5 --tag smoke
from pathlib import Path
stats = Path('results/norm_study/zinc_norm_batch_d4_smoke/0/test/stats.json')
if not stats.exists() or not stats.read_text(encoding='utf-8').strip():
    log = Path('logs/norm_study/zinc_norm_batch_d4_smoke.log')
    if log.exists():
        print('--- last 60 lines ---')
        print('\n'.join(log.read_text(errors='replace').splitlines()[-60:]))
    raise RuntimeError('Smoke test produced no stats.json. Fix it before the grid.')
print('One-cell smoke test OK:', stats)


In [ ]:
# Auto-restore environment. Kaggle may reset CWD to /kaggle/working on a
# reconnect, and on a full session recycle it can wipe /kaggle/working entirely.
import os
from pathlib import Path
REPO = Path('/kaggle/working/repo')
if not (REPO / 'main.py').exists():
    import subprocess
    subprocess.run(['git', 'clone',
                    'https://github.com/1589893076/CS5491-GNNPlus-NormStudy.git',
                    str(REPO)], check=True)
    print('repo was missing -> re-cloned from GitHub')
os.chdir(REPO)
assert (REPO / 'experiments' / 'run_norm_study.py').exists(), 'clone failed'

# Cell 6 — FULL GRID (20 cells). One session is enough at 30 epochs.
# --skip-existing makes this resumable after an interruption.
EPOCHS = 30            # keep identical to Cell 8
REPEAT = 3
NORMS = 'batch layer graph pair none'
DEPTHS = '4 8 12 20'
# If reusing a session that ran a DIFFERENT epoch budget, clear old results first:
# !rm -rf results/norm_study results/norm_study_summary.csv results/norm_study_summary.md
!python experiments/run_norm_study.py --repeat {REPEAT} --epochs {EPOCHS} --norms {NORMS} --depths {DEPTHS} --skip-existing

# Report progress across all 20 cells (scan results, not just this run's manifest).
import json
from pathlib import Path
root = REPO / 'results/norm_study'
norms = ['batch', 'layer', 'graph', 'pair', 'none']
depths = [4, 8, 12, 20]
done, missing = [], []
for n in norms:
    for d in depths:
        stats = root / ('zinc_norm_%s_d%d' % (n, d)) / '0' / 'test' / 'stats.json'
        (done if stats.exists() and stats.read_text(encoding='utf-8').strip() else missing).append('%s/d%d' % (n, d))
print('cells with seed-0 stats:', len(done), '/ 20')
print('missing:', missing)
assert len(done) == 20, 'Grid incomplete: run Cell 6 again (--skip-existing resumes).'
print('FULL GRID COMPLETE — continue to Cell 7.')


In [ ]:
# Auto-restore environment. Kaggle may reset CWD to /kaggle/working on a
# reconnect, and on a full session recycle it can wipe /kaggle/working entirely.
import os
from pathlib import Path
REPO = Path('/kaggle/working/repo')
if not (REPO / 'main.py').exists():
    import subprocess
    subprocess.run(['git', 'clone',
                    'https://github.com/1589893076/CS5491-GNNPlus-NormStudy.git',
                    str(REPO)], check=True)
    print('repo was missing -> re-cloned from GitHub')
os.chdir(REPO)
assert (REPO / 'experiments' / 'run_norm_study.py').exists(), 'clone failed'

# Cell 7 — aggregate results (run only when all 20 cells are done)
!python experiments/aggregate_norm_study.py
from pathlib import Path
import csv
summary = REPO / 'results/norm_study_summary.csv'
if not summary.exists():
    raise RuntimeError('No summary: the grid produced no stats.json. Go back and finish it.')
with summary.open(newline='', encoding='utf-8') as fh:
    rows = list(csv.DictReader(fh))
print('ZINC summary rows:', len(rows), '/ 20 planned cells')
if not rows:
    raise RuntimeError('Empty summary table. Do not download an empty result.')
if len(rows) < 20:
    print('WARNING: partial grid; the report will label it explicitly.')
from IPython.display import display, Markdown
md = REPO / 'results/norm_study_summary.md'
if md.exists():
    display(Markdown(md.read_text(encoding='utf-8')))


In [ ]:
# Auto-restore environment. Kaggle may reset CWD to /kaggle/working on a
# reconnect, and on a full session recycle it can wipe /kaggle/working entirely.
import os
from pathlib import Path
REPO = Path('/kaggle/working/repo')
if not (REPO / 'main.py').exists():
    import subprocess
    subprocess.run(['git', 'clone',
                    'https://github.com/1589893076/CS5491-GNNPlus-NormStudy.git',
                    str(REPO)], check=True)
    print('repo was missing -> re-cloned from GitHub')
os.chdir(REPO)
assert (REPO / 'experiments' / 'run_norm_study.py').exists(), 'clone failed'

# Cell 8 — rebuild both PDF reports with the SAME epoch budget as Cell 6.
EPOCHS = 30            # must match Cell 6
OUT = '/kaggle/working/cs5491_reports'
!python docs/build_reports.py --out-dir {OUT} --zinc-csv /kaggle/working/repo/results/norm_study_summary.csv --epochs {EPOCHS}
import os
for name in sorted(os.listdir(OUT)):
    print(name, os.path.getsize(os.path.join(OUT, name)), 'bytes')


In [ ]:
# Auto-restore environment. Kaggle may reset CWD to /kaggle/working on a
# reconnect, and on a full session recycle it can wipe /kaggle/working entirely.
import os
from pathlib import Path
REPO = Path('/kaggle/working/repo')
if not (REPO / 'main.py').exists():
    import subprocess
    subprocess.run(['git', 'clone',
                    'https://github.com/1589893076/CS5491-GNNPlus-NormStudy.git',
                    str(REPO)], check=True)
    print('repo was missing -> re-cloned from GitHub')
os.chdir(REPO)
assert (REPO / 'experiments' / 'run_norm_study.py').exists(), 'clone failed'

# Cell 9 — package everything for download
# Kaggle: right panel -> Output -> download cs5491_final.zip
import os, shutil
OUT = '/kaggle/working/cs5491_reports'
for src in (REPO / 'results/norm_study_summary.csv',
            REPO / 'results/norm_study_summary.md'):
    if os.path.exists(src):
        shutil.copyfile(src, os.path.join(OUT, os.path.basename(src)))
zip_base = '/kaggle/working/cs5491_final'
shutil.make_archive(zip_base, 'zip', OUT)
print('Created:', zip_base + '.zip', os.path.getsize(zip_base + '.zip'), 'bytes')


## Download & hand off

1. Right panel → **Output** → download cs5491_final.zip.
2. Extract it locally. Copy norm_study_summary.csv, norm_study_summary.md and the two
   PDFs into `F:\CS5491 小组项目\CS 5491\Final project\GNNPlus-main\results\`.
3. Tell the assistant — it will run docs/finish_project.py --epochs 30 to rebuild and
   verify the final PDFs (matching the budget you ran) and commit everything.

**Remember**: keep EPOCHS identical in Cell 6 and Cell 8 (both 30).
Save a Version at the end so results survive a runtime reset.
